<a href="https://colab.research.google.com/github/deekshdechamma/DL_skill_developer/blob/main/dl5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 5: Custom Batch Normalization and Layer Normalization
Forward/Backward Propagation
● Objective: Maximize training convergence speeds and stabilize internal covariate shifts by
building structural normalization layers with manual gradient derivations.
● Required Tech Stack: PyTorch (with Autograd disabled), NumPy.
● Task Description: Students must write functional modules for Batch Normalization and
Layer Normalization. They are required to derive and implement the forward passes
(calculating running mean and variance) and write the exact partial derivatives for
backpropagating gradients through the scale parameter , shift parameter , and
normalize transformations.

In [2]:
#Step 1 - Import libraries
import torch
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


In [4]:
#Step 2 — Create sample data
X = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [3.0, 6.0, 9.0],
    [4.0, 8.0, 12.0]
], dtype=torch.float32, requires_grad=False)

print("Input X:")
print(X)

print("\nShape:", X.shape)
print("Autograd enabled for X:", X.requires_grad)

Input X:
tensor([[ 1.,  2.,  3.],
        [ 2.,  4.,  6.],
        [ 3.,  6.,  9.],
        [ 4.,  8., 12.]])

Shape: torch.Size([4, 3])
Autograd enabled for X: False


In [5]:
#Step 3 — Custom Batch Normalization
class CustomBatchNorm:

    def __init__(self, num_features, eps=1e-5, momentum=0.1):

        self.eps = eps
        self.momentum = momentum

        # Scale parameter
        self.gamma = torch.ones(
            num_features,
            dtype=torch.float32,
            requires_grad=False
        )

        # Shift parameter
        self.beta = torch.zeros(
            num_features,
            dtype=torch.float32,
            requires_grad=False
        )

        # Running statistics
        self.running_mean = torch.zeros(num_features)

        self.running_var = torch.ones(num_features)

        self.cache = None

In [6]:
#Step 4 — Batch Normalization forward propagation
def batchnorm_forward(self, X, training=True):

    if training:

        # Number of samples
        N = X.shape[0]

        # Mean of each feature
        mean = X.mean(dim=0)

        # Variance of each feature
        var = ((X - mean) ** 2).mean(dim=0)

        # Standard deviation inverse
        inv_std = 1.0 / torch.sqrt(var + self.eps)

        # Normalize
        X_hat = (X - mean) * inv_std

        # Scale and shift
        Y = self.gamma * X_hat + self.beta

        # Update running statistics
        self.running_mean = (
            (1 - self.momentum) * self.running_mean
            + self.momentum * mean
        )

        self.running_var = (
            (1 - self.momentum) * self.running_var
            + self.momentum * var
        )

        # Save values required during backward propagation
        self.cache = (X, X_hat, mean, var, inv_std)

        return Y

    else:

        X_hat = (
            (X - self.running_mean)
            / torch.sqrt(self.running_var + self.eps)
        )

        Y = self.gamma * X_hat + self.beta

        return Y

In [7]:
#Step 5 — Batch Normalization backward propagation
def batchnorm_backward(self, dY):

    X, X_hat, mean, var, inv_std = self.cache

    N = X.shape[0]

    # Gradient with respect to beta
    dBeta = dY.sum(dim=0)

    # Gradient with respect to gamma
    dGamma = (dY * X_hat).sum(dim=0)

    # Gradient through scale operation
    dX_hat = dY * self.gamma

    # Manual gradient with respect to input
    dX = (1.0 / N) * inv_std * (
        N * dX_hat
        - dX_hat.sum(dim=0)
        - X_hat * (dX_hat * X_hat).sum(dim=0)
    )

    return dX, dGamma, dBeta

In [8]:
#Step 6 — Complete BatchNorm class
class CustomBatchNorm:

    def __init__(self, num_features, eps=1e-5, momentum=0.1):

        self.eps = eps
        self.momentum = momentum

        self.gamma = torch.ones(
            num_features,
            dtype=torch.float32,
            requires_grad=False
        )

        self.beta = torch.zeros(
            num_features,
            dtype=torch.float32,
            requires_grad=False
        )

        self.running_mean = torch.zeros(num_features)
        self.running_var = torch.ones(num_features)

        self.cache = None


    def forward(self, X, training=True):

        if training:

            mean = X.mean(dim=0)

            var = ((X - mean) ** 2).mean(dim=0)

            inv_std = 1.0 / torch.sqrt(
                var + self.eps
            )

            X_hat = (X - mean) * inv_std

            Y = self.gamma * X_hat + self.beta

            # Update running mean
            self.running_mean = (
                (1 - self.momentum)
                * self.running_mean
                + self.momentum
                * mean
            )

            # Update running variance
            self.running_var = (
                (1 - self.momentum)
                * self.running_var
                + self.momentum
                * var
            )

            self.cache = (
                X,
                X_hat,
                mean,
                var,
                inv_std
            )

            return Y

        else:

            X_hat = (
                (X - self.running_mean)
                / torch.sqrt(
                    self.running_var + self.eps
                )
            )

            Y = self.gamma * X_hat + self.beta

            return Y


    def backward(self, dY):

        X, X_hat, mean, var, inv_std = self.cache

        N = X.shape[0]

        # beta gradient
        dBeta = dY.sum(dim=0)

        # gamma gradient
        dGamma = (
            dY * X_hat
        ).sum(dim=0)

        # Gradient through scale parameter
        dX_hat = dY * self.gamma

        # Input gradient
        dX = (
            (1.0 / N)
            * inv_std
            * (
                N * dX_hat
                - dX_hat.sum(dim=0)
                - X_hat
                * (
                    dX_hat * X_hat
                ).sum(dim=0)
            )
        )

        return dX, dGamma, dBeta

In [9]:
#Step 7 — Test Batch Normalization forward pass
bn = CustomBatchNorm(num_features=3)

BN_output = bn.forward(X, training=True)

print("Original Input:")
print(X)

print("\nBatch Normalized Output:")
print(BN_output)

print("\nRunning Mean:")
print(bn.running_mean)

print("\nRunning Variance:")
print(bn.running_var)

Original Input:
tensor([[ 1.,  2.,  3.],
        [ 2.,  4.,  6.],
        [ 3.,  6.,  9.],
        [ 4.,  8., 12.]])

Batch Normalized Output:
tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])

Running Mean:
tensor([0.2500, 0.5000, 0.7500])

Running Variance:
tensor([1.0250, 1.4000, 2.0250])


In [10]:
#Step 8 — Test Batch Normalization backward pass
dY = torch.tensor([
    [1.0,  0.5, -0.5],
    [0.5, -1.0,  1.0],
    [1.5,  0.2,  0.5],
    [-0.5, 1.0,  1.5]
], dtype=torch.float32, requires_grad=False)

dX_bn, dGamma_bn, dBeta_bn = bn.backward(dY)

print("Gradient with respect to Input:")
print(dX_bn)

print("\nGradient with respect to Gamma:")
print(dGamma_bn)

print("\nGradient with respect to Beta:")
print(dBeta_bn)

Gradient with respect to Input:
tensor([[-0.1342,  0.3265, -0.0894],
        [-0.2683, -0.4651,  0.1938],
        [ 0.9391, -0.0492, -0.1193],
        [-0.5367,  0.1878,  0.0149]])

Gradient with respect to Gamma:
tensor([-1.5652,  1.2075,  2.4597])

Gradient with respect to Beta:
tensor([2.5000, 0.7000, 2.5000])


In [11]:
#Step 9 — Custom Layer Normalization
class CustomLayerNorm:

    def __init__(self, num_features, eps=1e-5):

        self.eps = eps

        self.gamma = torch.ones(
            num_features,
            dtype=torch.float32,
            requires_grad=False
        )

        self.beta = torch.zeros(
            num_features,
            dtype=torch.float32,
            requires_grad=False
        )

        self.cache = None


    def forward(self, X):

        # Mean across features
        mean = X.mean(
            dim=1,
            keepdim=True
        )

        # Variance across features
        var = (
            (X - mean) ** 2
        ).mean(
            dim=1,
            keepdim=True
        )

        # Inverse standard deviation
        inv_std = 1.0 / torch.sqrt(
            var + self.eps
        )

        # Normalize
        X_hat = (
            X - mean
        ) * inv_std

        # Scale and shift
        Y = (
            self.gamma * X_hat
            + self.beta
        )

        self.cache = (
            X,
            X_hat,
            mean,
            var,
            inv_std
        )

        return Y


    def backward(self, dY):

        X, X_hat, mean, var, inv_std = self.cache

        # Number of features
        D = X.shape[1]

        # Gradient of beta
        dBeta = dY.sum(dim=0)

        # Gradient of gamma
        dGamma = (
            dY * X_hat
        ).sum(dim=0)

        # Gradient through scaling
        dX_hat = (
            dY * self.gamma
        )

        # Manual input gradient
        dX = (
            (1.0 / D)
            * inv_std
            * (
                D * dX_hat
                - dX_hat.sum(
                    dim=1,
                    keepdim=True
                )
                - X_hat
                * (
                    dX_hat * X_hat
                ).sum(
                    dim=1,
                    keepdim=True
                )
            )
        )

        return dX, dGamma, dBeta

In [12]:
#Step 10 — Test Layer Normalization forward pass
ln = CustomLayerNorm(num_features=3)

LN_output = ln.forward(X)

print("Original Input:")
print(X)

print("\nLayer Normalized Output:")
print(LN_output)

Original Input:
tensor([[ 1.,  2.,  3.],
        [ 2.,  4.,  6.],
        [ 3.,  6.,  9.],
        [ 4.,  8., 12.]])

Layer Normalized Output:
tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])


In [13]:
#Step 11 — Test Layer Normalization backward pass
dX_ln, dGamma_ln, dBeta_ln = ln.backward(dY)

print("Gradient with respect to Input:")
print(dX_ln)

print("\nGradient with respect to Gamma:")
print(dGamma_ln)

print("\nGradient with respect to Beta:")
print(dBeta_ln)

Gradient with respect to Input:
tensor([[-0.1020,  0.2041, -0.1021],
        [ 0.3572, -0.7144,  0.3572],
        [ 0.1089, -0.2177,  0.1089],
        [-0.0510,  0.1021, -0.0510]])

Gradient with respect to Gamma:
tensor([-3.0619,  0.0000,  3.0619])

Gradient with respect to Beta:
tensor([2.5000, 0.7000, 2.5000])


In [14]:
#Step 12 — Compare BatchNorm and LayerNorm
print("=" * 60)
print("BATCH NORMALIZATION OUTPUT")
print("=" * 60)

print(BN_output)

print("\n" + "=" * 60)
print("LAYER NORMALIZATION OUTPUT")
print("=" * 60)

print(LN_output)

BATCH NORMALIZATION OUTPUT
tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])

LAYER NORMALIZATION OUTPUT
tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])


In [15]:
#Step 13 — Verify mean and variance
print("BatchNorm Mean across Batch:")
print(BN_output.mean(dim=0))

print("\nBatchNorm Variance across Batch:")
print(BN_output.var(dim=0, unbiased=False))


print("\nLayerNorm Mean for each Sample:")
print(LN_output.mean(dim=1))

print("\nLayerNorm Variance for each Sample:")
print(LN_output.var(dim=1, unbiased=False))

BatchNorm Mean across Batch:
tensor([0., 0., 0.])

BatchNorm Variance across Batch:
tensor([1.0000, 1.0000, 1.0000])

LayerNorm Mean for each Sample:
tensor([0., 0., 0., 0.])

LayerNorm Variance for each Sample:
tensor([1.0000, 1.0000, 1.0000, 1.0000])


In [16]:
#Step 14 — Simple parameter update
learning_rate = 0.01

# BatchNorm parameters
bn.gamma -= learning_rate * dGamma_bn
bn.beta -= learning_rate * dBeta_bn

# LayerNorm parameters
ln.gamma -= learning_rate * dGamma_ln
ln.beta -= learning_rate * dBeta_ln

print("Updated BatchNorm Gamma:")
print(bn.gamma)

print("\nUpdated BatchNorm Beta:")
print(bn.beta)

print("\nUpdated LayerNorm Gamma:")
print(ln.gamma)

print("\nUpdated LayerNorm Beta:")
print(ln.beta)

Updated BatchNorm Gamma:
tensor([1.0157, 0.9879, 0.9754])

Updated BatchNorm Beta:
tensor([-0.0250, -0.0070, -0.0250])

Updated LayerNorm Gamma:
tensor([1.0306, 1.0000, 0.9694])

Updated LayerNorm Beta:
tensor([-0.0250, -0.0070, -0.0250])


In [17]:
#Step 15 — Final demonstration
print("=" * 60)
print("TASK 5: CUSTOM NORMALIZATION")
print("=" * 60)

print("\nInput Shape:", X.shape)

print("\n--- Batch Normalization ---")
print("Output:")
print(BN_output)

print("\ndGamma:")
print(dGamma_bn)

print("\ndBeta:")
print(dBeta_bn)

print("\ndX:")
print(dX_bn)

print("\n--- Layer Normalization ---")
print("Output:")
print(LN_output)

print("\ndGamma:")
print(dGamma_ln)

print("\ndBeta:")
print(dBeta_ln)

print("\ndX:")
print(dX_ln)

print("\nAutograd used:", X.requires_grad)

print("\nTask 5 completed successfully!")

TASK 5: CUSTOM NORMALIZATION

Input Shape: torch.Size([4, 3])

--- Batch Normalization ---
Output:
tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])

dGamma:
tensor([-1.5652,  1.2075,  2.4597])

dBeta:
tensor([2.5000, 0.7000, 2.5000])

dX:
tensor([[-0.1342,  0.3265, -0.0894],
        [-0.2683, -0.4651,  0.1938],
        [ 0.9391, -0.0492, -0.1193],
        [-0.5367,  0.1878,  0.0149]])

--- Layer Normalization ---
Output:
tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])

dGamma:
tensor([-3.0619,  0.0000,  3.0619])

dBeta:
tensor([2.5000, 0.7000, 2.5000])

dX:
tensor([[-0.1020,  0.2041, -0.1021],
        [ 0.3572, -0.7144,  0.3572],
        [ 0.1089, -0.2177,  0.1089],
        [-0.0510,  0.1021, -0.0510]])

Autograd used: False

Task 5 completed successfully!
